In [ ]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Kernel

from qiskit.circuit.library import pauli_feature_map
from qiskit.primitives import StatevectorSampler

from qiskit_machine_learning.state_fidelities import ComputeUncompute
from qiskit_machine_learning.kernels import FidelityQuantumKernel

# ==========================
# LOAD DATA
# ==========================

dataset = pd.read_csv("../dataset/riemann_features.csv")

features = [
    "z_co_gram_lag_2",
    "z_gram",
    "z_gram_lag_1",
    "d_lag_13",
    "z_co_gram_lag_3",
    "z_co_gram_lag_1",
    "d_lag_14",
    "d_lag_1",
    "z_gram_lag_2",
    "d_lag_17"
]

X = dataset[features].values
y = dataset["distance"].values

X = X[:2000]
y = y[:2000]

split = int(0.8 * len(X))

X_train = X[:split]
X_test = X[split:]

y_train = y[:split]
y_test = y[split:]

# ==========================
# SCALE
# ==========================

scaler = MinMaxScaler(feature_range=(-1,1))

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# ==========================
# PCA (reduzir qubits)
# ==========================

pca = PCA(n_components=5)

X_train = pca.fit_transform(X_train)
X_test = pca.transform(X_test)

n_qubits = X_train.shape[1]

# ==========================
# QUANTUM KERNEL
# ==========================

feature_map = pauli_feature_map(
    feature_dimension=n_qubits,
    reps=4,
    entanglement="circular",
    paulis=["Z","ZZ","ZX"]
)

sampler = StatevectorSampler()

fidelity = ComputeUncompute(sampler=sampler)

quantum_kernel = FidelityQuantumKernel(
    feature_map=feature_map,
    fidelity=fidelity
)

# ==========================
# WRAPPER PARA SKLEARN
# ==========================

class QuantumKernelWrapper(Kernel):

    def __init__(self, quantum_kernel):
        self.quantum_kernel = quantum_kernel

    def __call__(self, X, Y=None, eval_gradient=False):

        if Y is None:
            K = self.quantum_kernel.evaluate(X)
        else:
            K = self.quantum_kernel.evaluate(X, Y)

        if eval_gradient:
            return K, np.zeros((K.shape[0], K.shape[1], 1))

        return K

    def diag(self, X):
        return np.ones(X.shape[0])

    def is_stationary(self):
        return False

kernel = QuantumKernelWrapper(quantum_kernel)

# ==========================
# GAUSSIAN PROCESS
# ==========================

gpr = GaussianProcessRegressor(
    kernel=kernel,
    alpha=1e-6,
    normalize_y=True
)

print("Treinando Quantum Gaussian Process...")

gpr.fit(X_train, y_train)

pred = gpr.predict(X_test)

# ==========================
# MÉTRICAS
# ==========================

rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)

print("\n===== Quantum Gaussian Process =====")
print("RMSE:", rmse)
print("R2:", r2)